In [31]:
from pathlib import Path
import re
import csv
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [21]:
load_dotenv()

user = os.getenv("MYSQL_USER")
password = os.getenv("MYSQL_PASSWORD")
host = os.getenv("MYSQL_HOST")
port = os.getenv("MYSQL_PORT")
database = os.getenv("MYSQL_DATABASE")

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
)

In [22]:
"""
This script is used to treat the MEROPS database and work with it. The DB includes
many .sql files necessary to create the peptidase-inhibitor-substrate relationships.
"""
merops_path = "../MEROPS/meropsweb121"
tables_path = "../MEROPS/tables"

In [14]:
query = """
SELECT code, organism, status
# SELECT *
FROM activity_status
WHERE organism = 'human'
"""

df = pd.read_sql(query, engine)

print(df.head())

      code organism    status
0  A01.001    human    active
1  A01.003    human    active
2  A01.004    human    active
3  A01.006    human  inactive
4  A01.007    human    active


In [15]:
# open peptidase_inhibitor_complex.sql and read it into a pandas dataframe
query = """
SELECT complex_id, peptidase_code, inhibitor_code, inhibitor_name, Ki
FROM peptidase_inhibitor_complex;
"""

df_complex = pd.read_sql(query, engine)

# print(df_complex.head())

peptidase_counts = df_complex['peptidase_code'].value_counts()
print(peptidase_counts[peptidase_counts > 1])
print(f"Number of peptidases with more than one inhibitor: {len(peptidase_counts[peptidase_counts > 1])}")
print(f"Number of peptidases with one inhibitor: {len(peptidase_counts[peptidase_counts == 1])}")

# Check if any inhibitor appears more than once
inhibitor_counts = df_complex['inhibitor_code'].value_counts()
print(inhibitor_counts[inhibitor_counts > 1])
print(f"Number of inhibitors with more than one peptidase: {len(inhibitor_counts[inhibitor_counts > 1])}")
print(f"Number of inhibitors with one peptidase: {len(inhibitor_counts[inhibitor_counts == 1])}")

peptidase_code
S01.151    257
A02.001    233
S01.217    136
S01.001    128
S01.216    113
          ... 
M43.005      2
S53.006      2
S01.342      2
M13.012      2
C13.001      2
Name: count, Length: 646, dtype: int64
Number of peptidases with more than one inhibitor: 646
Number of peptidases with one inhibitor: 345
inhibitor_code
J04.UPW    240
J29.001    208
J01.UPW    196
J29.002    190
J05.UPW    117
          ... 
I01.055      2
J16.408      2
I04.033      2
I16.010      2
I16.015      2
Name: count, Length: 706, dtype: int64
Number of inhibitors with more than one peptidase: 706
Number of inhibitors with one peptidase: 968


In [30]:
# Describe the following .sql files:
# substrate.sql, tree.sql, sequence.sql
# Print the column names and the number of rows for each table
tables = [
    # "substrate", 
    "tree", 
    # "sequence",
    ]

for table in tables:
    query = f"SELECT * FROM {table};"
    df_table = pd.read_sql(query, engine)
    print(f"Table: {table}")
    print(f"Columns: {df_table.columns.tolist()}")
    print(f"Number of rows: {len(df_table)}")
    print(df_table.head())

Table: tree
Columns: ['treeID', 'tree']
Number of rows: 432
  treeID                                               tree
0    A01  ((((((((MER0135288:1.50000,MER0425551:1.50000)...
1   A01A  (((((((((((((((((MER0383974:12.50000,MER001999...
2   A01B  (((((((((((((MER0015560:18.00000,MER0014127:18...
3    A02  ((((((((((((((MER0618126:27.00000,MER0225230:2...
4   A02A  ((((((((((((((((((((((MER0164941:0.00000,MER00...


In [ ]:
# 1. Cambia esta ruta por la carpeta donde están tus archivos .sql
carpeta = Path(r"C:\Users\guill\OneDrive - UNIR\ASSIGNATURES\TRABAJO_FIN_MASTER\DATABASES\MEROPS\meropsweb121")

# 2. Reconocer los tipos de datos habituales de MySQL
tipos_validos = {
    "int", "integer", "tinyint", "smallint", "mediumint", "bigint",
    "varchar", "char", "text", "tinytext", "mediumtext", "longtext",
    "enum", "set", "date", "datetime", "timestamp", "time", "year",
    "float", "double", "decimal", "numeric", "blob", "tinyblob",
    "mediumblob", "longblob", "binary", "varbinary", "json", "bool",
    "boolean", "bit"
}

# Extraer el nombre y tipo de una columna
def extraer_columna(linea):
    coincidencia = re.match(r"\s*`([^`]+)`\s+(.+)", linea)
    if not coincidencia:
        return None

    nombre = coincidencia.group(1)
    resto = coincidencia.group(2).strip()

    tipo_match = re.match(
        r"([A-Za-z]+(?:\s*\([^)]*\))?(?:\s+unsigned)?)",
        resto,
        re.IGNORECASE
    )
    if not tipo_match:
        return None

    tipo = tipo_match.group(1).strip()
    tipo_base = re.match(r"[A-Za-z]+", tipo).group(0).lower()

    if tipo_base not in tipos_validos:
        return None

    return {"nombre": nombre, "tipo": tipo}


# Analizar un archivo SQL línea por línea
def analizar_sql(archivo):
    tabla = None
    columnas = []
    dentro_create = False

    patron_create = re.compile(
        r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?`([^`]+)`",
        re.IGNORECASE
    )

    with archivo.open("r", encoding="latin-1", errors="replace") as f:
        for linea in f:
            if not dentro_create:
                coincidencia = patron_create.search(linea)
                if not coincidencia:
                    continue

                tabla = coincidencia.group(1)
                dentro_create = True

                # Si hay definición de columnas en la misma línea
                if "(" in linea:
                    linea = linea.split("(", 1)[1]
                    resto = linea.split(")", 1)[0]
                    columna = extraer_columna(resto)
                    if columna:
                        columnas.append(columna)
                continue

            # Fin de la sentencia CREATE TABLE
            if re.match(r"\s*\)\s*(?:ENGINE|TYPE|;|$)", linea, re.IGNORECASE):
                break

            columna = extraer_columna(linea)
            if columna:
                columnas.append(columna)

    return tabla, columnas


# 3. Recorrer los archivos y generar el catálogo
catalogo = []

for archivo in sorted(carpeta.glob("*.sql")):
    tabla, columnas = analizar_sql(archivo)
    tamano_mb = archivo.stat().st_size / (1024 * 1024)

    catalogo.append({
        "archivo": archivo.name,
        "tabla": tabla or "No detectada",
        "columnas": ", ".join(c["nombre"] for c in columnas),
        "tipos": ", ".join(
            f"{c['nombre']}: {c['tipo']}" for c in columnas
        ),
        "tamano_MB": round(tamano_mb, 3)
    })

# 4. Mostrar el catálogo
for item in catalogo:
    print(f"\nArchivo: {item['archivo']}")
    print(f"Tabla: {item['tabla']}")
    print(f"Tamaño: {item['tamano_MB']} MB")
    print(f"Columnas: {item['columnas']}")
    print(f"Tipos: {item['tipos']}")

# 5. Guardar los resultados en un CSV
salida = carpeta / "catalogo_merops.csv"

with salida.open("w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=[
        "archivo", "tabla", "columnas", "tipos", "tamano_MB"
    ])
    escritor.writeheader()
    escritor.writerows(catalogo)

print(f"\nCatálogo guardado en: {salida}")


Archivo: activity_status.sql
Tabla: activity_status
Tamaño: 0.053 MB
Columnas: code, organism, status, comment, reference
Tipos: code: varchar(8), organism: enum('human','mouse'), status: enum('active','inactive','putative'), comment: text, reference: tinytext

Archivo: aligned_sequence.sql
Tabla: aligned_sequence
Tamaño: 166.377 MB
Columnas: MERNO, level, aligned_sequence, MERNUM, family
Tipos: MERNO: varchar(10), level: enum('family','subfamily'), aligned_sequence: mediumtext, MERNUM: int(6) unsigned, family: tinytext

Archivo: alternative_id.sql
Tabla: alternative_id
Tamaño: 314.143 MB
Columnas: sequence_id, alternative_id, alternative_id_count, type, embln_notes, genomeID, selected_for_protein
Tipos: sequence_id: varchar(20), alternative_id: varchar(50), alternative_id_count: int(10) unsigned, type: enum('uniprot','embln','emblp','trembl','pir'), embln_notes: tinytext, genomeID: int(11), selected_for_protein: enum('Yes','No')

Archivo: architecture.sql
Tabla: architecture
Tamaño: 

In [ ]:
"""
NOTAS DE LAS BBDD.
Tablas que contiene información relevante:
    - peptidase_inhibitor_complex: 
        · contiene las relaciones entre peptidasas e inhibidores, incluyendo el Ki.
        · columnas: complex_id, peptidase_code, inhibitor_code, inhibitor_name, Ki
        · se deberían eliminar aquellas sin Ki
    - activity_status:
        · contiene información sobre el estado de actividad de las peptidasas.
        · eliminar aquellas con status: inactive
"""